<!-- course-header -->
# Lesson 01: Python for AI work

*Companion notebook for lesson 01. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/01-python-for-ai.html) (it works well on a phone) or [offline](../lessons/01-python-for-ai.html). Worked solutions are in `solutions/01-python-for-ai-solutions.ipynb`; try each exercise first.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/01-python-for-ai.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

The Python you actually need for AI, then NumPy for fast array maths and pandas plus Matplotlib for loading, cleaning and plotting real data.

**Runs on:** Any computer (CPU).

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy pandas matplotlib

## Python essentials for AI code

### Data structures you will use daily

In [ ]:
# lists: ordered, mutable. Batches of examples, histories of messages.
messages = [{"role": "user", "content": "Hi"}]
messages.append({"role": "assistant", "content": "Hello!"})

# dicts: key to value. Configs, JSON payloads, token-to-id vocabularies.
config = {"model": "small", "lr": 3e-4, "epochs": 10}
lr = config.get("lr", 1e-3)            # default if the key is missing

# tuples: fixed, immutable. Shapes, (input, label) pairs.
shape = (32, 128)                      # batch of 32, 128 features

# sets: unique items, fast membership tests. Vocabularies, de-duplication.
seen = set()
for word in "the cat sat on the mat".split():
    seen.add(word)
print(len(seen))                        # 5

### Comprehensions

Comprehensions build collections in one readable line. You will see them everywhere in data-preparation code.

In [ ]:
lengths = [len(w) for w in ["attention", "is", "all"]]            # [9, 2, 3]
vocab = {word: i for i, word in enumerate(sorted(seen))}            # word -> id
long_words = [w for w in seen if len(w) > 2]                        # filter
pairs = [(x, y) for x in range(3) for y in range(2)]                # nested loops

### Functions with type hints and defaults

Type hints do not change how Python runs, but they make code self-documenting, let your editor catch mistakes, and they are how libraries such as Pydantic and FastAPI (which you will use in parts VI and VII) know what data to expect.

In [ ]:
def chunk_text(text: str, size: int = 500, overlap: int = 50) -> list[str]:
    """Split text into overlapping chunks of roughly `size` characters."""
    if overlap >= size:
        raise ValueError("overlap must be smaller than size")
    step = size - overlap
    return [text[i : i + size] for i in range(0, len(text), step)]

chunks = chunk_text("lorem ipsum " * 200, size=300)
print(len(chunks), len(chunks[0]))

### Classes and dataclasses

Models, datasets and agents are naturally objects: some state plus methods that act on it. For plain data containers, `dataclass` writes the boilerplate for you.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Document:
    id: str
    text: str
    metadata: dict = field(default_factory=dict)   # never use a bare {} default

    def preview(self, n: int = 60) -> str:
        return self.text[:n] + ("…" if len(self.text) > n else "")

doc = Document(id="manual-3", text="Calibrate the spectrometer before each campaign.")
print(doc)            # Document(id='manual-3', text='Calibrate ...', metadata={})
print(doc.preview(20))

> **The mutable default trap**
>
> `def f(x, cache={})` creates **one** dictionary when the function is defined and shares it across every call. Use `None` as the default and create the object inside, or `field(default_factory=dict)` in a dataclass.

### Generators: data that does not fit in memory

A generator produces values one at a time with `yield`. Training data and log files are often too large to load at once; generators let you stream them.

In [ ]:
def read_jsonl(path: str):
    import json
    with open(path) as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def batched(iterable, n: int):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == n:
            yield batch
            batch = []
    if batch:
        yield batch

# for batch in batched(read_jsonl("train.jsonl"), 32): ...

### Errors and retries

Calls to model APIs fail: networks drop, rate limits bite. Handle the errors you expect and let unexpected ones surface.

In [ ]:
import time, random

def with_retries(fn, attempts: int = 5, base: float = 0.5):
    for i in range(attempts):
        try:
            return fn()
        except (TimeoutError, ConnectionError) as e:
            if i == attempts - 1:
                raise
            wait = base * 2 ** i + random.uniform(0, 0.1)   # exponential backoff with jitter
            print(f"attempt {i + 1} failed ({e}); retrying in {wait:.2f}s")
            time.sleep(wait)

## NumPy: thinking in arrays

### Creating arrays and reading shapes

In [ ]:
import numpy as np

a = np.array([1.0, 2.0, 3.0])          # shape (3,)
M = np.arange(12).reshape(3, 4)        # shape (3, 4): 3 rows, 4 columns
Z = np.zeros((2, 3))
rng = np.random.default_rng(0)         # the modern, reproducible way to get randomness
X = rng.normal(size=(100, 5))          # 100 samples, 5 features

print(M.shape, M.ndim, M.dtype)        # (3, 4) 2 int64
print(M[1, 2])                         # row 1, column 2 -> 6
print(M[:, 0])                         # first column -> [0 4 8]
print(M[M > 5])                        # boolean mask -> [ 6  7  8  9 10 11]

> **Shapes are your best debugging tool**
>
> When something breaks in ML code, print the shapes first. Most bugs are a tensor with shape `(32, 1)` where you expected `(32,)`, or rows and columns swapped. Get in the habit of writing the expected shape as a comment next to each line.

### Vectorisation: stop writing loops over numbers

A Python `for` loop over a million numbers runs at Python speed. The same operation written as array maths runs in compiled code, often 50 to 200 times faster.

In [ ]:
import time
x = rng.normal(size=1_000_000)

t = time.perf_counter()
total = 0.0
for v in x:
    total += v * v
loop_time = time.perf_counter() - t

t = time.perf_counter()
total_vec = np.sum(x * x)              # or x @ x
vec_time = time.perf_counter() - t
print(f"loop {loop_time:.3f}s, vectorised {vec_time:.4f}s")

### Broadcasting

Broadcasting lets NumPy combine arrays of different shapes without copying data. The rule: compare shapes from the right; two dimensions are compatible if they are equal or one of them is 1.

In [ ]:
X = rng.normal(size=(100, 5))          # (100, 5)
mu = X.mean(axis=0)                    # (5,)  mean of each column
sigma = X.std(axis=0)                  # (5,)
X_std = (X - mu) / sigma               # (100, 5) - (5,) -> broadcast across rows
print(X_std.mean(axis=0).round(6))     # ~0 for every column
print(X_std.std(axis=0).round(6))      # ~1 for every column

### Axis, reductions and matrix multiplication

In [ ]:
M = np.arange(6).reshape(2, 3)         # [[0 1 2], [3 4 5]]
M.sum()                                # 15        everything
M.sum(axis=0)                          # [3 5 7]   collapse rows, one value per column
M.sum(axis=1)                          # [3 12]    collapse columns, one value per row

W = rng.normal(size=(3, 4))
out = M @ W                            # (2, 3) @ (3, 4) -> (2, 4)

### Hands-on: vectorise a distance computation

You have 1,000 query vectors `Q` of shape `(1000, 64)` and 5,000 database vectors `D` of shape `(5000, 64)`. Compute the matrix of Euclidean distances, shape `(1000, 5000)`, **without any Python loops**. Then find, for each query, the index of its nearest database vector. Hint: expand $\lVert q - d\rVert^2 = \lVert q\rVert^2 - 2\,q\cdot d + \lVert d\rVert^2$.

In [ ]:
# Your code here

## pandas and Matplotlib: real data is messy

### A week of sensor data

To keep the course offline, we generate a realistic dataset: a week of five-minute readings from an air-quality station, with a daily temperature cycle, morning and evening NO₂ peaks from traffic, some missing values and a few faulty spikes.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
times = pd.date_range("2026-06-01", periods=7 * 24 * 12, freq="5min")   # one week, every 5 minutes
hours = times.hour + times.minute / 60
temp = 18 + 6 * np.sin(2 * np.pi * (hours - 9) / 24) + rng.normal(0, 0.4, len(times))
no2 = (12 + 10 * np.exp(-((hours - 8) ** 2) / 4)       # morning rush hour
          + 8 * np.exp(-((hours - 18) ** 2) / 5)        # evening rush hour
          + rng.normal(0, 1.5, len(times)))
df = pd.DataFrame({"time": times, "temp_c": temp, "no2_ppb": no2})

gaps = rng.choice(len(df), size=60, replace=False)      # simulate dropouts
df.loc[gaps, "no2_ppb"] = np.nan
spikes = rng.choice(len(df), size=8, replace=False)     # simulate faulty readings
df.loc[spikes, "temp_c"] += rng.choice([-25, 30], size=8)
df.to_csv("sensor.csv", index=False)

### Look before you model

In [ ]:
df = pd.read_csv("sensor.csv", parse_dates=["time"])
df.info()                      # column types and non-null counts
print(df.describe().round(2))  # min / max reveal the spikes: temp max is ~54 °C
print(df.isna().sum())         # 60 missing NO2 values

### Clean: outliers, gaps, resampling

In [ ]:
# 1. Flag outliers as points far from a rolling median (robust to the spikes themselves)
med = df["temp_c"].rolling(13, center=True, min_periods=1).median()
outlier = (df["temp_c"] - med).abs() > 5
print("outliers:", outlier.sum())                     # 8
df.loc[outlier, "temp_c"] = np.nan

# 2. Fill short gaps by interpolating in time (but never invent long stretches)
df = df.set_index("time").interpolate(method="time", limit=6)

# 3. Resample to hourly means
hourly = df.resample("1h").mean()

# 4. Group by hour of day to see the daily cycle
profile = df.groupby(df.index.hour)["no2_ppb"].mean()
print(profile.idxmax())                               # 8: the morning peak

### Plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=False)
hourly["temp_c"].plot(ax=axes[0], color="tab:red")
axes[0].set_ylabel("Temperature (°C)")
profile.plot(ax=axes[1], marker="o")
axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel("Mean NO₂ (ppb)")
fig.tight_layout()
plt.show()

### Hands-on: extend the analysis

Using the cleaned data:

1. Add a column `is_weekend` (the week starts on a Monday) and compare the mean NO₂ on weekdays versus weekends. In our synthetic data there is no difference; in real data there would be. Why?
2. Compute a 1-hour rolling mean of NO₂ and plot it over the raw series for one day.
3. Find the three hours of the week with the highest mean NO₂ using `nlargest`.

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/01-python-for-ai.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.